# Análise de Benchmark — io_uring vs prim_file

Comparação estatística de throughput e latência P99 entre o caminho de
escrita padrão do RabbitMQ (`prim_file`/`file:write`) e a integração
`io_uring` desenvolvida neste projeto.

**Executado via:** `bench_statistical.sh` ou `bench_consumer.sh`  
**Parâmetros:** lidos das variáveis de ambiente `BENCH_*`

In [ ]:
import os, csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

# ── parâmetros ─────────────────────────────────────────────────────────────
BASELINE_CSV  = os.environ.get('BENCH_BASELINE_CSV',  'bench-results/baseline.csv')
IOURING_CSV   = os.environ.get('BENCH_IOURING_CSV',   'bench-results/iouring.csv')
RUNS          = int(os.environ.get('BENCH_RUNS',       '30'))
DURATION      = int(os.environ.get('BENCH_DURATION',   '10'))
PRODUCERS     = int(os.environ.get('BENCH_PRODUCERS',  '4'))
CONSUMERS     = int(os.environ.get('BENCH_CONSUMERS',  '0'))
MSG_SIZE      = int(os.environ.get('BENCH_SIZE',       '1024'))
CONSUMER_MODE = os.environ.get('BENCH_CONSUMER_MODE', '0') == '1'
OUT_DIR       = Path(BASELINE_CSV).parent

print(f'Baseline CSV : {BASELINE_CSV}')
print(f'io_uring CSV : {IOURING_CSV}')
print(f'Modo         : {"consumer" if CONSUMER_MODE else "escrita"}')
print(f'Runs / dur.  : {RUNS} × {DURATION}s')
print(f'Msg size     : {MSG_SIZE} B')

## 1. Leitura dos dados

In [ ]:
def load_csv(path: str) -> tuple[np.ndarray, np.ndarray | None]:
    tp, p99 = [], []
    with open(path) as f:
        for row in csv.reader(f):
            if not row:
                continue
            try:
                tp.append(float(row[0]))
                if len(row) >= 2:
                    p99.append(float(row[1]))
            except ValueError:
                pass
    return np.array(tp), (np.array(p99) if p99 else None)

b_tp, b_p99 = load_csv(BASELINE_CSV)
i_tp, i_p99 = load_csv(IOURING_CSV)

if CONSUMER_MODE:
    b_p99 = i_p99 = None

print(f'Baseline  : {len(b_tp)} runs  |  média {b_tp.mean():,.0f} msg/s')
print(f'io_uring  : {len(i_tp)} runs  |  média {i_tp.mean():,.0f} msg/s')
if b_p99 is not None:
    print(f'P99 baseline  : {b_p99.mean():,.0f} µs')
    print(f'P99 io_uring  : {i_p99.mean():,.0f} µs')

## 2. Análise Estatística

Usamos o **teste t de Welch** (variâncias independentes) para comparar as duas
populações. Reportamos também o **d de Cohen** como medida de tamanho de efeito
e o **intervalo de confiança 95%** via bootstrap (10 000 amostras).

In [ ]:
def cohens_d(a: np.ndarray, b: np.ndarray) -> float:
    pooled = np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
    return (a.mean() - b.mean()) / pooled if pooled else 0.0

def ci95(data: np.ndarray) -> tuple[float, float]:
    rng = np.random.default_rng(42)
    boot = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(10_000)]
    return float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

def effect_label(d: float) -> str:
    d = abs(d)
    if d < 0.2: return 'negligível'
    if d < 0.5: return 'pequeno'
    if d < 0.8: return 'médio'
    return 'grande'

def sig_stars(p: float) -> str:
    if p < 0.001: return '*** (p < 0.001)'
    if p < 0.01:  return '**  (p < 0.01)'
    if p < 0.05:  return '*   (p < 0.05)'
    return 'ns  (p ≥ 0.05)'

# Throughput
t_tp, p_tp = stats.ttest_ind(i_tp, b_tp, equal_var=False)
d_tp       = cohens_d(i_tp, b_tp)
ci_b_tp    = ci95(b_tp)
ci_i_tp    = ci95(i_tp)
delta_tp   = i_tp.mean() - b_tp.mean()
delta_tp_p = (i_tp.mean() / b_tp.mean() - 1) * 100

tp_label  = 'Consumer Throughput' if CONSUMER_MODE else 'Throughput'
hdr_label = (f'{CONSUMERS} consumers' if CONSUMER_MODE else f'{PRODUCERS} produtores')

sep = '═' * 62
print(f'\n{sep}')
print(f'  ANÁLISE ESTATÍSTICA — {RUNS} runs × {DURATION}s')
print(f'  {hdr_label} | {MSG_SIZE} B | persistent | io_uring vs prim_file')
print(sep)
print(f'\n  ┌── {tp_label.upper()} (msg/s) ──────────────────────────────────┐')
print(f'  {"Baseline (prim_file)":<24} {b_tp.mean():>12,.1f}   CI₉₅=[{ci_b_tp[0]:,.0f}, {ci_b_tp[1]:,.0f}]')
print(f'  {"io_uring":<24} {i_tp.mean():>12,.1f}   CI₉₅=[{ci_i_tp[0]:,.0f}, {ci_i_tp[1]:,.0f}]')
print(f'  {"Δ médio":<24} {delta_tp:>+12,.1f}   ({delta_tp_p:+.1f}%)')
print(f'  Welch t({len(b_tp)+len(i_tp)-2:.0f}) = {t_tp:.3f},  p = {p_tp:.4e}  {sig_stars(p_tp)}')
print(f'  Cohen\'s d = {d_tp:.3f}  ({effect_label(d_tp)})')

stats_res = dict(t_tp=t_tp, p_tp=p_tp, d_tp=d_tp, t_p99=None, p_p99=None, d_p99=None)

if b_p99 is not None and len(b_p99):
    t_p99, p_p99  = stats.ttest_ind(i_p99, b_p99, equal_var=False)
    d_p99         = cohens_d(b_p99, i_p99)
    ci_b_p99      = ci95(b_p99)
    ci_i_p99      = ci95(i_p99)
    delta_p99     = i_p99.mean() - b_p99.mean()
    delta_p99_p   = (i_p99.mean() / b_p99.mean() - 1) * 100
    stats_res.update(dict(t_p99=t_p99, p_p99=p_p99, d_p99=d_p99))

    print(f'\n  ┌── CONFIRM P99 LATENCY (µs) ─────────────────────────────────┐')
    print(f'  {"Baseline (prim_file)":<24} {b_p99.mean():>12,.1f} µs  CI₉₅=[{ci_b_p99[0]:,.0f}, {ci_b_p99[1]:,.0f}]')
    print(f'  {"io_uring":<24} {i_p99.mean():>12,.1f} µs  CI₉₅=[{ci_i_p99[0]:,.0f}, {ci_i_p99[1]:,.0f}]')
    print(f'  {"Δ médio":<24} {delta_p99:>+12,.1f}   ({delta_p99_p:+.1f}%)')
    print(f'  Welch t({len(b_p99)+len(i_p99)-2:.0f}) = {t_p99:.3f},  p = {p_p99:.4e}  {sig_stars(p_p99)}')
    print(f'  Cohen\'s d = {d_p99:.3f}  ({effect_label(d_p99)})')

print(f'\n{sep}')

## 3. Boxplots com jitter e anotação de significância

In [ ]:
BLUE   = '#2196F3'
ORANGE = '#FF5722'
GRAY   = '#9E9E9E'

has_p99 = b_p99 is not None and len(b_p99) > 0
ncols   = 2 if has_p99 else 1
fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 6))
if ncols == 1:
    axes = [axes]

fig.suptitle(
    f'io_uring vs prim_file  —  {RUNS} runs × {DURATION}s'
    f'  |  {hdr_label}  |  {MSG_SIZE} B  |  persistent',
    fontsize=12, fontweight='bold', y=1.01,
)

def style_box(bplot, colors):
    for patch, c in zip(bplot['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.75)
    for med in bplot['medians']:
        med.set_color('black')
        med.set_linewidth(2)

def sig_bracket(ax, p, y, x0, x1):
    stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    h = y * 0.03
    ax.plot([x0, x0, x1, x1], [y, y + h, y + h, y], lw=1.2, c='black')
    ax.text((x0 + x1) / 2, y + h * 1.1, stars, ha='center', va='bottom',
            fontsize=12, fontweight='bold')

jitter = np.random.default_rng(0).uniform(-0.08, 0.08, max(len(b_tp), len(i_tp)))

# Throughput
ax = axes[0]
bplot = ax.boxplot(
    [b_tp / 1_000, i_tp / 1_000],
    tick_labels=['Baseline\n(prim_file)', 'io_uring'],
    patch_artist=True, widths=0.5, showfliers=True,
    flierprops=dict(marker='o', markersize=4, alpha=0.5),
)
style_box(bplot, [GRAY, BLUE])
ax.set_ylabel('Throughput (k-msg/s)', fontsize=11)
ax.set_title(tp_label, fontsize=12, fontweight='bold')
ax.scatter(1 + jitter[:len(b_tp)], b_tp / 1_000, s=18, c=GRAY, alpha=0.6, zorder=5)
ax.scatter(2 + jitter[:len(i_tp)], i_tp / 1_000, s=18, c=BLUE, alpha=0.6, zorder=5)
y_top = max(b_tp.max(), i_tp.max()) / 1_000 * 1.08
sig_bracket(ax, stats_res['p_tp'], y_top, 1, 2)
ax.set_ylim(top=y_top * 1.12)
ax.grid(axis='y', alpha=0.3)

# P99 latência (opcional)
if has_p99:
    ax = axes[1]
    bplot = ax.boxplot(
        [b_p99 / 1_000, i_p99 / 1_000],
        tick_labels=['Baseline\n(prim_file)', 'io_uring'],
        patch_artist=True, widths=0.5, showfliers=True,
        flierprops=dict(marker='o', markersize=4, alpha=0.5),
    )
    style_box(bplot, [GRAY, ORANGE])
    ax.set_ylabel('Confirm p99 Latency (ms)', fontsize=11)
    ax.set_title('Latência p99 (menor = melhor)', fontsize=12, fontweight='bold')
    ax.scatter(1 + jitter[:len(b_p99)], b_p99 / 1_000, s=18, c=GRAY,   alpha=0.6, zorder=5)
    ax.scatter(2 + jitter[:len(i_p99)], i_p99 / 1_000, s=18, c=ORANGE, alpha=0.6, zorder=5)
    y_top = max(b_p99.max(), i_p99.max()) / 1_000 * 1.08
    sig_bracket(ax, stats_res['p_p99'], y_top, 1, 2)
    ax.set_ylim(top=y_top * 1.12)
    ax.grid(axis='y', alpha=0.3)

legend_handles = [
    mpatches.Patch(fc=GRAY, label=f'Baseline  (n={len(b_tp)})'),
    mpatches.Patch(fc=BLUE, label=f'io_uring  (n={len(i_tp)})'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, -0.05), fontsize=10)

annot = [f'Welch t-test (two-sided)',
         f'Throughput:  t={stats_res["t_tp"]:.2f},  p={stats_res["p_tp"]:.2e},  d={stats_res["d_tp"]:.2f}']
if has_p99 and stats_res['t_p99'] is not None:
    annot.append(f'p99 latency: t={stats_res["t_p99"]:.2f},  p={stats_res["p_p99"]:.2e},  d={stats_res["d_p99"]:.2f}')
fig.text(0.5, -0.12, '\n'.join(annot), ha='center', va='top',
         fontsize=9, family='monospace',
         bbox=dict(boxstyle='round', fc='lightyellow', ec='goldenrod', alpha=0.8))

plt.tight_layout()
suffix = 'consumer_' if CONSUMER_MODE else ''
boxplot_path = OUT_DIR / f'{suffix}benchmark_boxplots.png'
plt.savefig(boxplot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico salvo em: {boxplot_path}')

## 4. Série temporal — estabilidade das runs

In [ ]:
nrows = 2 if has_p99 else 1
fig, axes = plt.subplots(nrows, 1, figsize=(13, 4 * nrows), sharex=True)
if nrows == 1:
    axes = [axes]
runs_x = np.arange(1, len(b_tp) + 1)

ax = axes[0]
ax.plot(runs_x, b_tp / 1_000, 'o-', c=GRAY, label='Baseline', lw=1.5, ms=5)
ax.plot(runs_x, i_tp / 1_000, 's-', c=BLUE, label='io_uring', lw=1.5, ms=5)
ax.axhline(b_tp.mean() / 1_000, c=GRAY, ls='--', lw=1, alpha=0.7)
ax.axhline(i_tp.mean() / 1_000, c=BLUE, ls='--', lw=1, alpha=0.7)
ax.set_ylabel('Throughput (k-msg/s)')
ax.set_title(f'{tp_label} por run', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

if has_p99:
    ax = axes[1]
    ax.plot(runs_x, b_p99 / 1_000, 'o-', c=GRAY,   label='Baseline', lw=1.5, ms=5)
    ax.plot(runs_x, i_p99 / 1_000, 's-', c=ORANGE, label='io_uring', lw=1.5, ms=5)
    ax.axhline(b_p99.mean() / 1_000, c=GRAY,   ls='--', lw=1, alpha=0.7)
    ax.axhline(i_p99.mean() / 1_000, c=ORANGE, ls='--', lw=1, alpha=0.7)
    ax.set_ylabel('Confirm p99 (ms)')
    ax.set_title('Confirm p99 latency por run', fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Run #')
fig.suptitle(
    f'Série temporal — {RUNS} runs × {DURATION}s  |  {hdr_label}  |  {MSG_SIZE} B',
    fontweight='bold',
)
plt.tight_layout()
timeseries_path = OUT_DIR / f'{suffix}benchmark_timeseries.png'
plt.savefig(timeseries_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Série temporal salva em: {timeseries_path}')

## 4b. Separate PDFs for LaTeX (no legend, English axis labels)

In [ ]:
suffix = 'consumer_' if CONSUMER_MODE else ''

# ── Throughput PDF ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
runs_x = np.arange(1, len(b_tp) + 1)

ax.plot(runs_x, b_tp / 1_000, 'o-', c=GRAY, lw=1.5, ms=5, label='Baseline (prim_file)')
ax.plot(runs_x, i_tp / 1_000, 's-', c=BLUE, lw=1.5, ms=5, label='io_uring')
ax.axhline(b_tp.mean() / 1_000, c=GRAY, ls='--', lw=1, alpha=0.7)
ax.axhline(i_tp.mean() / 1_000, c=BLUE, ls='--', lw=1, alpha=0.7)
ax.set_xlabel('Run', fontsize=12)
ax.set_ylabel('Throughput (k-msg/s)', fontsize=12)
ax.legend(fontsize=11, loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
tp_pdf = OUT_DIR / f'{suffix}timeseries_throughput.pdf'
plt.savefig(tp_pdf, format='pdf', bbox_inches='tight')
plt.show()
print(f'Throughput PDF: {tp_pdf}')

# ── Latency PDF ─────────────────────────────────────────────────────────────
if has_p99:
    fig, ax = plt.subplots(figsize=(7, 3.5))

    ax.plot(runs_x, b_p99 / 1_000, 'o-', c=GRAY,   lw=1.5, ms=5, label='Baseline (prim_file)')
    ax.plot(runs_x, i_p99 / 1_000, 's-', c=ORANGE, lw=1.5, ms=5, label='io_uring')
    ax.axhline(b_p99.mean() / 1_000, c=GRAY,   ls='--', lw=1, alpha=0.7)
    ax.axhline(i_p99.mean() / 1_000, c=ORANGE, ls='--', lw=1, alpha=0.7)
    ax.set_xlabel('Run', fontsize=12)
    ax.set_ylabel('Confirm p99 latency (ms)', fontsize=12)
    ax.legend(fontsize=11, loc='upper right')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    lat_pdf = OUT_DIR / f'{suffix}timeseries_latency.pdf'
    plt.savefig(lat_pdf, format='pdf', bbox_inches='tight')
    plt.show()
    print(f'Latency PDF: {lat_pdf}')

## 5. Distribuição (histograma + KDE)

In [ ]:
from scipy.stats import gaussian_kde

ncols = 2 if has_p99 else 1
fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 5))
if ncols == 1:
    axes = [axes]

def kde_plot(ax, data, color, label):
    kde = gaussian_kde(data, bw_method='scott')
    x = np.linspace(data.min() * 0.97, data.max() * 1.03, 300)
    ax.fill_between(x, kde(x), alpha=0.25, color=color)
    ax.plot(x, kde(x), color=color, lw=2, label=label)
    ax.axvline(data.mean(), color=color, ls='--', lw=1.5, alpha=0.8)

ax = axes[0]
kde_plot(ax, b_tp / 1_000, GRAY, f'Baseline  μ={b_tp.mean()/1_000:.1f}k')
kde_plot(ax, i_tp / 1_000, BLUE, f'io_uring  μ={i_tp.mean()/1_000:.1f}k')
ax.set_xlabel('Throughput (k-msg/s)')
ax.set_ylabel('Densidade')
ax.set_title(f'Distribuição — {tp_label}', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

if has_p99:
    ax = axes[1]
    kde_plot(ax, b_p99 / 1_000, GRAY,   f'Baseline  μ={b_p99.mean()/1_000:.1f}ms')
    kde_plot(ax, i_p99 / 1_000, ORANGE, f'io_uring  μ={i_p99.mean()/1_000:.1f}ms')
    ax.set_xlabel('Confirm p99 (ms)')
    ax.set_ylabel('Densidade')
    ax.set_title('Distribuição — Latência p99 (menor = melhor)', fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle(
    f'Distribuição KDE — {hdr_label}  |  {MSG_SIZE} B  |  {RUNS} runs',
    fontweight='bold',
)
plt.tight_layout()
kde_path = OUT_DIR / f'{suffix}benchmark_kde.png'
plt.savefig(kde_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'KDE salvo em: {kde_path}')

## 6. Resumo final

In [ ]:
print('\n' + '═' * 62)
print('  RESUMO')
print('═' * 62)
print(f'  Throughput  : {delta_tp_p:+.1f}%  ({sig_stars(p_tp)})  d={d_tp:.2f} ({effect_label(d_tp)})')
if has_p99 and stats_res['p_p99'] is not None:
    print(f'  P99 latência: {delta_p99_p:+.1f}%  ({sig_stars(p_p99)})  d={d_p99:.2f} ({effect_label(d_p99)})')
print('═' * 62)
print(f'  Arquivos gerados em: {OUT_DIR}/')